In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
intonations = ['Frustrated but Polite', 'Overwhelmed Newbie', 'DIY, Curious and Tech Savy']

In [66]:
issue = "The water reservoir is leaking, causing a mess on the countertop."
solution_step = """
1. "Unplug the appliance from the power source to ensure safety before inspecting the water reservoir."
2. "Remove the water reservoir from the appliance carefully."
3. "Check for any visible cracks or damage on the reservoir. If found, replace the reservoir."
4. "Inspect the seal or gasket on the water reservoir. If it appears worn or damaged, replace it."
5. "Ensure that the reservoir is seated properly back into the appliance. Misalignment can cause leaks."
6. "Fill the reservoir with water and place it back in the appliance."
7. "Plug the appliance back in and monitor for any signs of leaking."
"""

conversation_prompt = f"""
Synthetic Conversation Generation:
Task: Generate a realistic, synthetic multi-turn conversation (maximum 15 turns) between a user and HomeHelper (the assistant) grounded in the  {issue} and {solution_step}.

- Generate three sets of conversations per scenario with human set up in different intonations:
    - Frustrated but Polite
    - Overwhelmed Newbie
    - DIY, Curious and Tech Savy
- Each conversation is multi-turn and of maximum 15 turns
- Use natural dialogue; keep it human-like
- Avoid generic or shallow answers - be specific and grounded.
- End when the issue is resolved.
- Format the conversation clearly:
User: [User message]
HomeHelper: [Assistant response]
"""


In [ ]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# Define root path of product PDFs
products_path = "/home/sgaur2/cse247-HomeHelper-Sigma/ProcessedProducts"
index_folder = "indexes"

# Get all PDF file paths recursively
def get_all_pdf_paths(start_dir):
    pdf_paths = []
    for root, _, files in os.walk(start_dir):
        for name in files:
            if name.lower().endswith(".pdf"):
                pdf_paths.append(os.path.join(root, name))
    return pdf_paths

# Create vector store for a single PDF
def create_index_for_pdf(pdf_path, index_path, log_path="resources/skipped_pdfs.txt"):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    if not documents:
        print(f"⚠️ Skipping: {pdf_path} (no extractable text)")
        with open(log_path, "a") as log:
            log.write(f"{pdf_path} - no text extracted\n")
        return None

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    if not chunks:
        print(f"⚠️ Skipping: {pdf_path} (no chunks after split)")
        with open(log_path, "a") as log:
            log.write(f"{pdf_path} - no chunks after split\n")
        return None

    embedding = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(chunks, embedding, persist_directory=index_path)
    vectorstore.persist()

    return vectorstore

# === Combine index creation and metadata collection ===
scenarios = {}
pdf_paths = get_all_pdf_paths(products_path)

for pdf_path in pdf_paths:
    filename = os.path.basename(pdf_path)
    index_name = f"{filename.replace('.pdf', '')}_index"
    index_path = os.path.join(index_folder, index_name)

    if not os.path.exists(index_path):
        print(f"Creating index for: {filename}")
        success = create_index_for_pdf(pdf_path, index_path)
    else:
        print(f"Index already exists for: {filename}")
        success = True  # treat as successful if already built

    if success:
        company_name = os.path.basename(os.path.dirname(pdf_path))
        product_name = os.path.basename(os.path.dirname(os.path.dirname(pdf_path)))
        scenarios[index_name] = {
            "filename": filename,
            "product_name": product_name,
            "company_name": company_name
        }

# === Print summary ===
print(f"\n✅ Total indexed PDFs: {len(scenarios)}")
sample_key = list(scenarios.keys())[0]
print(f"\n🧾 Sample entry [{sample_key}]:\n{scenarios[sample_key]}")

In [11]:
import re
import json

scenario_prompt = f"""
You are HomeHelper, an AI assistant that helps users troubleshoot and repair household appliances using product manuals.

Based on the attached manual, generate a list of **10 distinct issues** a user might encounter with the appliance. Ensure the issues are meaningful—neither too trivial nor overly technical.

- The issues must be **grounded in the manual only**. Do not invent or infer problems that are not clearly supported by the content.
- **6 issues** should be **specific to the appliance model** (use the model name as mentioned in the manual).
- **4 issues** can be more **general** and applicable to appliances of this type.
- Avoid repetition; each issue should highlight a unique user pain point.

⚠️ Output rules:
- Do not bold or format any text with asterisks.
- Do not include an introduction or conclusion.
- Just list the issues in two clearly labeled sections.

📌 Example format:

Model-Specific Issues (KF7070):
1. The clock display fails to light up even when the appliance is powered.
2. Auto-brew function doesn't activate at the programmed time.

Generic Issues for this appliance:
1. Brewed coffee tastes weak even when using the correct amount of grounds.
"""

def query_pdf(index_name, user_query):
    index_path = os.path.join("indexes", index_name)
    embedding = OpenAIEmbeddings()
    vectorstore = Chroma(persist_directory=index_path, embedding_function=embedding)

    qa = RetrievalQA.from_chain_type(
        llm=ChatOpenAI(model="gpt-4o"),
        retriever=vectorstore.as_retriever()
    )

    return qa.run(user_query)


for index_name in scenarios.keys():
    response = query_pdf(index_name, scenario_prompt)
    issues = re.findall(r'\d+\.\s+(.*)', response)
    scenarios[index_name]["issues"] = issues

# write scenarios to a json file with identation
with open("scenarios.json", "w") as f:
    json.dump(scenarios, f, indent=4)
    

In [ ]:
import json

with open("resources/scenarios.json", "r") as f:
    scenarios = json.load(f)

for index, value in scenarios.items():
    print(f"\n📄 Processing PDF index: {index}")
    
    # Prepare to store all solution steps
    value["solution_steps"] = []

    for issue in value["issues"]:
        # Dynamically generate the prompt with the current issue
        solution_steps_prompt = f"""
            You are HomeHelper, an AI assistant built to help users troubleshoot and repair household appliances.

            Using only the content from the attached manual, provide a clear, step-by-step solution for the following issue:
            {issue}

            Instructions:
            - Each step must be grounded strictly in the manual. Do not infer or invent solutions.
            - Quote the manual verbatim wherever possible to maintain accuracy.
            - Be specific and actionable.
            - Do not include any introduction or summary.
            - Do not use bold, asterisks, or any special formatting.

            Output only the numbered solution steps.
            """

        response = query_pdf(index, solution_steps_prompt)
        value["solution_steps"].append({
            "issue": issue,
            "steps": response
        })
        
# write the solution steps to a json file
with open("resources/solution_steps.json", "w") as f:
    json.dump(scenarios, f, indent=2)


In [ ]:
# Creating a sqlite database home_helper.db
import sqlite3

# Create or connect to SQLite DB
conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

# Create the table
cursor.execute('''
CREATE TABLE IF NOT EXISTS appliance_scenarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    appliance_name TEXT NOT NULL,
    company_name TEXT NOT NULL,
    product_name TEXT NOT NULL,
    scenario TEXT NOT NULL,
    solution_steps TEXT NOT NULL,
    human_intonations TEXT NOT NULL,
    conversation TEXT NOT NULL
)
''')

In [ ]:
import sqlite3
import json

# Load the JSON file
with open("solution_steps.json", "r") as f:
    scenarios = json.load(f)

# Connect to the database
conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

# Define the 3 intonations
intonations = [
    "Frustrated but Polite",
    "Overwhelmed Newbie",
    "DIY, Curious and Tech Savy"
]

# Iterate and insert
for index_name, value in scenarios.items():
    filename = value["filename"]
    company_name = value["company_name"]
    product_name = value["product_name"]
    try:
        steps_list = value["solution_steps"]  # already contains {"issue": ..., "steps": ...}
    except:
        print(index_name)
        break

    for step_obj in steps_list:
        issue = step_obj["issue"]
        steps = step_obj["steps"]

        for tone in intonations:
            cursor.execute('''
                INSERT INTO appliance_scenarios (
                    appliance_name,
                    company_name,
                    product_name,
                    scenario,
                    solution_steps,
                    human_intonations,
                    conversation
                ) VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (
                filename,
                company_name,
                product_name,
                issue,
                steps,
                tone,
                ""  # empty conversation for now
            ))

# Finalize
conn.commit()
conn.close()
print("✅ All entries inserted from solution_steps.")

In [ ]:
import sqlite3
import openai
import os

client = openai.OpenAI()
# Connect to the SQLite database
conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

# Fetch rows that have empty conversation field (you can modify as needed)
cursor.execute("SELECT id, appliance_name, company_name, product_name, scenario, solution_steps, human_intonations FROM appliance_scenarios WHERE conversation = '' and product_name in ('Blender-FoodProcessor', 'CoffeeMaker', 'Dishwasher', 'Dryer')")
rows = cursor.fetchall()

for row in rows:
    row_id, appliance_name, company_name, product_name, scenario, solution_steps, intonation = row

    # Patch the prompt
    conversation_prompt = f"""
You are HomeHelper, an AI assistant designed to help users troubleshoot and repair household appliances.

Task: Generate a realistic, synthetic multi-turn conversation (maximum 15 turns) between a user and HomeHelper (the assistant) based on the following troubleshooting scenario and solution steps.

- Intonation: {intonation}
- Scenario: {scenario}
- Steps: {solution_steps}

Instructions:
- Use natural dialogue; keep it human-like and true to the tone described.
- Avoid generic or shallow answers — ground every turn in the provided solution steps.
- End the conversation when the issue is resolved or fully guided.
- Do not use any formatting like bold or asterisks.
- Format the conversation in **single-spaced lines** using exactly **one line break (\\n)** between each turn.
- Do not insert any extra blank lines or paragraph breaks.
- Format the conversation clearly:
User: [User message]
HomeHelper: [Assistant response]
- Do not add any introduction or conclusion.
- Do not quote or mention “scenario” or “solution steps” directly — let them reflect naturally in the conversation.
"""

    # Call GPT API
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": conversation_prompt}],
            temperature=0.4,
        )

        conversation_text = response.choices[0].message.content

        # Save conversation back into DB
        cursor.execute(
            "UPDATE appliance_scenarios SET conversation = ? WHERE id = ?",
            (conversation_text, row_id)
        )
        
        conn.commit()
        print(f"✅ Conversation saved for row {row_id}")
        

    except Exception as e:
        print(f"❌ Error for row {row_id}: {e}")

cursor.close()
conn.close()

In [ ]:
# Clear the table
import sqlite3

conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

# Delete all rows from the table
cursor.execute("DELETE FROM appliance_scenarios")

# Optional: reset autoincrement ID
try:
    cursor.execute("DELETE FROM sqlite_sequence WHERE name='appliance_scenarios'")
except sqlite3.OperationalError as e:
    print("Could not reset autoincrement:", e)

conn.commit()
conn.close()

print("🧹 Table cleared successfully.")

In [ ]:
# Count the number of rows in the table
import sqlite3

conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

cursor.execute("SELECT count(*) FROM appliance_scenarios where conversation = ''")
row_count = cursor.fetchone()[0]

print(f"Number of rows: {row_count}")

cursor.close()
conn.close()

In [ ]:
# Fetch a row from the table
import sqlite3

conn = sqlite3.connect("home_helper.db")
cursor = conn.cursor()

conversation_text = ""

cursor.execute("SELECT * FROM appliance_scenarios")
rows = cursor.fetchall()

for row in rows:
    print(row)
    break


cursor.close()
conn.close()

In [ ]:
import sqlite3
import json

# Path to your SQLite database
db_path = 'home_helper.db'

# Output JSONL file path
output_path = 'resources/home_helper_dataset.jsonl'

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Define the table name
table_name = 'appliance_scenarios'

# Fetch all rows from the table
cursor.execute(f"SELECT * FROM {table_name}")
rows = cursor.fetchall()

# Retrieve column names from the cursor description
column_names = [description[0] for description in cursor.description]

# Open the output file in write mode
with open(output_path, 'w', encoding='utf-8') as f:
    for row in rows:
        # Create a dictionary mapping column names to row values
        record = dict(zip(column_names, row))
        # Serialize the dictionary to a JSON-formatted string
        json_record = json.dumps(record, ensure_ascii=False)
        # Write the JSON string to the file, followed by a newline
        
        
        f.write(json_record + '\n')

# Close the database connection
conn.close()

print(f"✅ Exported {len(rows)} records to {output_path}")